# DTW-RL routing evaluations and handovers

Notebook ini membaca lima CSV DTW-RL. Evaluasi routing dihitung dari `RC,W=...,N=...,C=...`, sedangkan handover terkonfirmasi dihitung dari `HO,n,delay_ms=...`.

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv
import re
from collections import Counter
import pandas as pd

# Ganti lima path berikut sesuai lokasi CSV di Google Drive.
CSV_FILES = [
    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/DTWRLmob5Interval101.csv',
    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/DTWRLmob5Interval102.csv',
    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/DTWRLmob5Interval103.csv',
    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/DTWRLmob5Interval104.csv',
    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/DTWRLmob5Interval105.csv',

]
OUTPUT_DIR = Path('/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/analisis_handover_5runs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert len(CSV_FILES) == 5, 'Masukkan tepat lima file CSV.'
for filename in CSV_FILES:
    if not Path(filename).is_file():
        print('File belum ditemukan:', filename)

RC_RE = re.compile(r'\bRC,W=(\d+),N=(\d+),C=(\d+)')
HO_RE = re.compile(r'\bHO,(\d+),delay_ms=([0-9]+(?:\.[0-9]+)?)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
def read_rows(path):
    # Mendukung message yang dikutip maupun message dengan koma tanpa kutip.
    with open(path, 'r', encoding='utf-8', errors='replace', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader, None)
        if header is None or [x.strip().lower() for x in header[:3]] != ['time', 'node', 'message']:
            raise ValueError(f'{path} harus memiliki kolom time,node,message')
        for row_number, row in enumerate(reader, start=2):
            if len(row) < 3:
                continue
            try:
                node = int(row[1].strip())
            except ValueError:
                continue
            yield row_number, row[0].strip(), node, ','.join(row[2:]).strip()

def analyze_file(filename):
    path = Path(filename)
    run = path.stem
    rc_rows = 0
    evaluation_total = 0
    evaluations_by_node = Counter()
    handovers_by_node = Counter()
    evaluations_by_window = Counter()
    handover_events = []
    diagnostics = Counter()

    for row_number, timestamp, node, message in read_rows(path):
        rc_match = RC_RE.search(message)
        if rc_match:
            window, logged_node, count = map(int, rc_match.groups())
            rc_rows += 1
            evaluation_total += count
            evaluations_by_node[logged_node] += count
            evaluations_by_window[window] += count

        ho_match = HO_RE.search(message)
        if ho_match:
            counter, delay_ms = ho_match.groups()
            handovers_by_node[node] += 1
            handover_events.append({
                'run': run, 'source_row': row_number, 'time': timestamp,
                'node': node, 'handover_counter': int(counter),
                'delay_ms': float(delay_ms),
            })

        diagnostics['controller_window_records'] += int('TC,W=' in message)
        diagnostics['energy_window_records'] += int('TE,W=' in message)
        diagnostics['timer_boundary_records'] += int('TB,W=' in message)
        diagnostics['core_parent_change_lines'] += int('Changed preferred parent' in message)
        diagnostics['set_preferred_parent_lines'] += int('rpl_set_preferred_parent' in message)
        diagnostics['dio_tx'] += int('MMDEW-CTRL: Tx DIO' in message)
        diagnostics['dao_tx'] += int('MMDEW-CTRL: Tx DAO' in message)
        diagnostics['dis_tx'] += int('MMDEW-CTRL: Tx DIS' in message)
        diagnostics['local_repairs'] += int('Starting a local instance repair' in message)

    handover_total = len(handover_events)
    summary = {
        'run': run, 'source_file': str(path),
        'evaluation_counter_status': 'available' if evaluation_total else 'unavailable',
        'rc_log_rows': rc_rows,
        'routing_evaluations_rank_via_parent': evaluation_total,
        'confirmed_handover_events': handover_total,
        'handover_per_evaluation': handover_total / evaluation_total if evaluation_total else pd.NA,
        'handovers_per_100_evaluations': 100 * handover_total / evaluation_total if evaluation_total else pd.NA,
        'mean_handover_delay_ms': sum(x['delay_ms'] for x in handover_events) / handover_total if handover_total else pd.NA,
        **diagnostics,
    }

    nodes = sorted(set(evaluations_by_node) | set(handovers_by_node))
    per_node = [{
        'run': run, 'node': node,
        'routing_evaluations': evaluations_by_node.get(node, 0),
        'confirmed_handovers': handovers_by_node.get(node, 0),
        'handover_per_evaluation': (handovers_by_node.get(node, 0) / evaluations_by_node[node]
                                    if evaluations_by_node.get(node, 0) else pd.NA),
    } for node in nodes]

    windows = [{
        'run': run, 'window': window, 'routing_evaluations': count
    } for window, count in sorted(evaluations_by_window.items())]
    return summary, per_node, handover_events, windows


In [45]:
summaries = []
per_node_all = []
handover_events_all = []
windows_all = []

for filename in CSV_FILES:
    summary, per_node, events, windows = analyze_file(filename)
    summaries.append(summary)
    per_node_all.extend(per_node)
    handover_events_all.extend(events)
    windows_all.extend(windows)
    print(f"{summary['run']}: evaluations={summary['routing_evaluations_rank_via_parent']}, "
          f"handovers={summary['confirmed_handover_events']}, "
          f"HO/evaluation={summary['handover_per_evaluation']}")

summary_df = pd.DataFrame(summaries)
per_node_df = pd.DataFrame(per_node_all)
handover_events_df = pd.DataFrame(handover_events_all)
windows_df = pd.DataFrame(windows_all)

summary_df.to_csv(OUTPUT_DIR / 'dtwrl_handover_eval_summary.csv', index=False)
per_node_df.to_csv(OUTPUT_DIR / 'dtwrl_handover_eval_per_node.csv', index=False)
handover_events_df.to_csv(OUTPUT_DIR / 'dtwrl_handover_events.csv', index=False)
windows_df.to_csv(OUTPUT_DIR / 'dtwrl_routing_eval_windows.csv', index=False)

display(summary_df)
print(f'\nHasil tersimpan di: {OUTPUT_DIR}')


DTWRLmob5Interval101: evaluations=0, handovers=3051, HO/evaluation=<NA>
DTWRLmob5Interval102: evaluations=0, handovers=3210, HO/evaluation=<NA>
DTWRLmob5Interval103: evaluations=0, handovers=2742, HO/evaluation=<NA>
DTWRLmob5Interval104: evaluations=0, handovers=4048, HO/evaluation=<NA>
DTWRLmob5Interval105: evaluations=0, handovers=3120, HO/evaluation=<NA>


,run,source_file,evaluation_counter_status,rc_log_rows,routing_evaluations_rank_via_parent,confirmed_handover_events,handover_per_evaluation,handovers_per_100_evaluations,mean_handover_delay_ms,controller_window_records,energy_window_records,timer_boundary_records,core_parent_change_lines,set_preferred_parent_lines,dio_tx,dao_tx,dis_tx,local_repairs
0,DTWRLmob5Interval101,/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3...,unavailable,0,0,3051,<NA>,<NA>,17391.434612,30993,0,0,1062,1150,3313,2528,74,59
1,DTWRLmob5Interval102,/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3...,unavailable,0,0,3210,<NA>,<NA>,19165.535514,26660,0,0,956,1041,2940,2381,72,56
2,DTWRLmob5Interval103,/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3...,unavailable,0,0,2742,<NA>,<NA>,24913.881109,33758,0,0,1013,1094,2975,2364,71,52
3,DTWRLmob5Interval104,/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3...,unavailable,0,0,4048,<NA>,<NA>,15486.229496,33159,0,0,1593,1716,3858,3572,101,94
4,DTWRLmob5Interval105,/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3...,unavailable,0,0,3120,<NA>,<NA>,19892.144231,29095,0,0,1416,1545,3836,3225,106,100



Hasil tersimpan di: /content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/Mobility5/DTWRL/10/analisis_handover_5runs
